# Data Science · Woche 6 · Praktikum: Der ETL-Prozess

**B.Sc. Wirtschaftsinformatik · Hochschule Ruhr West · Do 12.11.2026**

In der Vorlesung lief der ETL-Prozess an einer Supermarktkette: Gewinn mit Eis und Temperatur, zwei Filialen, eine
Woche. Heute gehen Sie denselben Weg mit
echten Daten: rund 100 000 Olist-Bestellungen aus drei Quellen. Am Ende steht eine Tabelle in `app_daten/projekt.db` –
genau das verlangt Schritt 4 Ihres Projekts.

**Die Frage:** In welchen Bundesstaaten kommen Olist-Bestellungen am häufigsten zu spät – und wie viel Umsatz steckt in
den verspäteten Bestellungen, **in Euro zum Kurs am Kauftag**?

**Warum zum Kurs am Kauftag?** Olist verkauft in Real (R$). Ein Euro kostete am 23.02.2017 3,25 R$, am 14.09.2018
4,88 R$ – die Hälfte mehr. Rechnet man eine Bestellung vom September 2018 mit dem Kurs vom Februar 2017 um, ist ihr
Euro-Betrag um die Hälfte zu hoch. Jede Bestellung braucht deshalb den Kurs ihres Kauftags – und wir eine Quelle mit dem
Kurs jedes Tages.

| Wann | Was |
|---|---|
| im Praktikum | gemeinsam am Beamer: der Fahrplan, dann Block E (Aufgabe 1 bis 3) |
| zu Hause | Block T und L (Aufgabe 4 bis 10) in Ihrem Tempo, dazu die ⭐-Aufgabe Ihrer Projekt-Karte. Die Musterlösung steht in Moodle – zum Vergleichen und wenn Sie festhängen. |

## Fahrplan

| Aufgabe | Problem | Was Sie üben | Python | Im Projekt |
|---|---|---|---|---|
| **E** | **Extrahieren** | | | |
| 1 | Bestellungen liegen in einer Datenbank | Zeilen und Spalten auswählen | `sqlite3.connect` · `pd.read_sql` | Ihre Webseite liest so aus `projekt.db` |
| 2 | Kunden und Verkäufer als CSV-Datei | erst ansehen, dann lesen | `pd.read_csv(…, dtype=…)` | Karten 3–6: zweite Quelle als Datei |
| 3 | Euro-Kurse gibt es im Web, über eine API | URL mit Parametern, Statuscode, JSON → Tabelle | `requests.get` · `.json()` | Karte 1 Wetter · Karte 2 Euribor |
| **T** | **Transformieren** | | | |
| 4 | Zeiten als Text, Lücken, „zu spät“ fehlt | Typen, neue Spalten, Lücken; Tag mit Tag vergleichen | `pd.to_datetime` · `.dt.date` · `dropna` | alle Karten |
| 5 | Kunden und Verkäufer in eine Tabelle | untereinander: gleiche Spalten, Kennspalte | `rename` · `pd.concat` | wenn eine Quelle in Teilen kommt |
| 6 | Staat und Umsatz an jede Bestellung | nebeneinander; wird rechts nachgeschlagen, dort jeder Schlüssel nur einmal – feiner? erst aggregieren | `merge(…, validate=…)` · `groupby` · `assert` | Karte 4: erst je Inserat · alle: `assert` |
| 7 | mehrere Bewertungen je Bestellung | Annahme prüfen, keine Bestellung verlieren | `validate` · `how='left'` | alle Karten |
| 8 | Kurs am Kauftag – aber nicht an jedem Tag | Schlüssel vereinheitlichen, Zeilen ohne Partner finden, Lücken füllen | `indicator=True` · `on=[…, …]` · `fillna` | Karte 1 Tag und Stunde · Karte 2 Jahr und Monat |
| **L** | **Laden** | | | |
| 9 | Ergebnis in die Projekt-Datenbank | zweimal laden, gleiches Ergebnis | `to_sql(…, if_exists=…)` | alle Karten: `projekt.db` |
| 10 | die Frage beantworten | in SQL gruppieren und aggregieren | `pd.read_sql` · `GROUP BY` | alle Karten: Zahlen für die Webseite |
| **⭐** | **für einzelne Karten** | | | |
| ⭐ 11 | dieselben Kurse als CSV | gleiche Quelle, anderes Format | `pd.read_csv(…, sep=…, decimal=…)` | Karte 2: `format=csvdata` |
| ⭐ 12 | Region von Kunde und Verkäufer | dieselbe Nachschlagetabelle zweimal | `rename` · zweimal `merge` · `assert` | Karte 6: Abhol- und Zielzone · Karte 3 |

## So ist jede Aufgabe gebaut

1. **📖 Worum es geht** – das Problem und warum der Schritt nötig ist.
2. **Jetzt Sie** – eine Code-Zelle. Wo `= None` steht, gehört Ihr Ergebnis hin; `# TODO` sagt, was herauskommen soll.
3. **Check** – darunter steht ✓ richtig · ✗ prüfen (mit dem erwarteten Wert) · … noch offen.
4. **💡 Tipp** – zugeklappt, nennt die Funktion.

Die Aufgaben bauen aufeinander auf. Hängen Sie fest, nehmen Sie die Zeile aus der Musterlösung und machen Sie weiter.
**Pflicht** sind Aufgabe 1 bis 10. Die Aufgaben mit **⭐** brauchen Sie, wenn Ihre Projekt-Karte sie nennt.

## Spickzettel

| | Schritt | Python |
|---|---|---|
| E | Datenbank | `verbindung = sqlite3.connect('datei.db')` · `pd.read_sql(sql, verbindung)` |
| E | Datei | erst ansehen · `pd.read_csv(…, sep=…, decimal=…, encoding=…, dtype={'spalte': str})` |
| E | API | `antwort = requests.get(url, timeout=60)` · `antwort.raise_for_status()` · `pd.DataFrame(antwort.json())` |
| T | Typen, Namen | `pd.to_datetime(…, format=…)` · `.dt.date` · `.dt.year` · `astype(float)` · `rename(columns={…})` |
| T | Lücken | `dropna(subset=[…])` · `fillna(…)` |
| T | aggregieren | `groupby('schlüssel')['spalte'].sum().reset_index()` |
| T | untereinander | Kennspalte anlegen, gleiche Spaltennamen, `pd.concat([a, b])` |
| T | rechts prüfen | `rechts['schlüssel'].duplicated().sum()` · `drop_duplicates(subset=…, keep=…)` |
| T | nebeneinander | `links.merge(rechts, on=…, how=…, validate=…, indicator=True)` |
| T | prüfen | `assert len(neu) == len(alt)` · `tabelle['_merge'].value_counts()` |
| L | laden | `tabelle.to_sql('name', verbindung, if_exists=…, index=False)` |
| L | abfragen | `pd.read_sql('SELECT … GROUP BY …', verbindung)` |

## 0 · Setup (1 Minute)

**Datei → Kopie in Drive speichern**, dann die Zelle unten ausführen. Sie holt drei Dateien aus dem Kurs-Repository in
Ihren Arbeitsordner: die Datenbank `olist.db` und zwei CSV-Dateien. Erwartet: drei Zeilen „bereit“, darunter `Setup ok`.

*Falls das Laden scheitert:* aus https://github.com/ZhuosHub/Data_Science_HRW_Praktikum (Ordner `data/olist`) die drei Dateien herunterladen, links über das
Ordner-Symbol hochladen und die Zelle erneut ausführen.

In [ ]:
import os
import shutil
import sqlite3                      # Datenbanken im SQLite-Format
import urllib.request

import pandas as pd                 # Tabellen
import requests                     # Anfragen an eine API

REPO_URL = 'https://raw.githubusercontent.com/ZhuosHub/Data_Science_HRW_Praktikum/main/data/'


def zahl(n):                        # für die Checks: 96478 → '96 478'
    return f'{n:,}'.replace(',', ' ')


for name in ['olist.db', 'olist_customers_dataset.csv', 'olist_sellers_dataset.csv']:
    if not os.path.exists(name):
        if os.path.exists('../data/olist/' + name):                       # im Ordner des Repositorys
            shutil.copy('../data/olist/' + name, name)
        else:                                                           # in Colab: von GitHub holen
            urllib.request.urlretrieve(REPO_URL + 'olist/' + name, name)
    print(f'{name}: bereit ({os.path.getsize(name) / 1e6:.1f} MB)')
print('Setup ok')

---
# Block E · Extrahieren

Drei Quellen, drei Wege – wie in der Vorlesung: Datenbank, Datei, Web (hier über eine API). Beim Extrahieren
entscheiden Sie auch, **was** Sie holen: welche Zeilen, welche Spalten, welchen Zeitraum.

## Aufgabe 1 · Datenbank: Bestellungen, Positionen, Bewertungen

**📖 Worum es geht.** Die Bestelldaten liegen in der Datenbank `olist.db`. Olist selbst gibt sie als CSV-Dateien heraus;
für den Kurs haben wir daraus eine SQLite-Datenbank mit drei Tabellen gebaut (ohne die Freitexte der Bewertungen) – so
liegen Bestelldaten in einem Shop-System. Für die Frage brauchen wir nur **gelieferte** Bestellungen: Nur bei ihnen lässt sich sagen, ob sie zu spät
kamen. Und nur diese Spalten:

| Tabelle | Spalten | Zeilen |
|---|---|---|
| `orders` | `order_id`, `customer_id`, `order_purchase_timestamp` (gekauft), `order_delivered_customer_date` (geliefert), `order_estimated_delivery_date` (geplant) | nur `order_status = 'delivered'` |
| `order_items` | `order_id`, `price`, `freight_value` – eine Zeile je Artikel | alle |
| `order_reviews` | `order_id`, `review_score` (1 bis 5 Sterne) | alle |

In [ ]:
# ===== Jetzt Sie (1) =====
olist = None          # TODO: Verbindung zu olist.db
bestellungen = None   # TODO: gelieferte Bestellungen, die fünf Spalten aus der Tabelle oben
positionen = None     # TODO: order_items, drei Spalten
bewertungen = None    # TODO: order_reviews, zwei Spalten

# --- Check ---
spalten_1 = {'order_id', 'customer_id', 'order_purchase_timestamp', 'order_delivered_customer_date',
             'order_estimated_delivery_date'}
if bestellungen is None or positionen is None or bewertungen is None:
    print('… noch offen')
elif not spalten_1.issubset(bestellungen.columns):
    print(f'✗ prüfen: in bestellungen fehlen Spalten: {sorted(spalten_1 - set(bestellungen.columns))}')
elif len(bestellungen) != 96478:
    print(f"✗ prüfen: erwartet 96 478 gelieferte Bestellungen, gefunden {zahl(len(bestellungen))} – WHERE order_status = 'delivered'")
elif not {'order_id', 'price', 'freight_value'}.issubset(positionen.columns) or len(positionen) != 112650:
    print('✗ prüfen: positionen – erwartet 112 650 Zeilen mit order_id, price, freight_value')
elif not {'order_id', 'review_score'}.issubset(bewertungen.columns) or len(bewertungen) != 99224:
    print('✗ prüfen: bewertungen – erwartet 99 224 Zeilen mit order_id, review_score')
else:
    print('✓ richtig · 96 478 gelieferte Bestellungen, 112 650 Positionen, 99 224 Bewertungen. Die Zeiten sind noch Text.')

<details><summary><b>💡 Tipp</b> · erst selbst versuchen, dann aufklappen</summary>
<p><code>pd.read_sql(sql, olist)</code> nimmt die Abfrage als Text. Die Spalten stehen hinter <code>SELECT</code>, die Zeilen wählt <code>WHERE</code> – wie in der Vorlesung bei der Kasse.</p>
</details>

## Aufgabe 2 · Datei: Kunden und Verkäufer

**📖 Worum es geht.** Kunden und Verkäufer liegen als CSV-Dateien vor. Wie in der Vorlesung: **erst ansehen, dann
lesen.** Die Zelle unten zeigt den Anfang der Kundendatei, so wie er in der Datei steht. Welches Trennzeichen? Gibt es
Dezimalzahlen? Muss eine Null vorn erhalten bleiben? In Woche 5 haben Sie die Postleitzahl nachträglich mit `zfill`
repariert – besser ist: gleich beim Lesen richtig.

In [ ]:
# ===== Erst ansehen: die ersten drei Zeilen der Datei =====
with open('olist_customers_dataset.csv', encoding='utf-8') as datei:
    for _ in range(3):
        print(datei.readline(), end='')

In [ ]:
# ===== Jetzt Sie (2) =====
kunden = None       # TODO: olist_customers_dataset.csv lesen – die Postleitzahl bleibt Text
verkaeufer = None   # TODO: olist_sellers_dataset.csv genauso (Spalte seller_zip_code_prefix)

# --- Check ---
if kunden is None or verkaeufer is None:
    print('… noch offen')
elif len(kunden) != 99441 or len(verkaeufer) != 3095:
    print(f'✗ prüfen: erwartet 99 441 Zeilen in kunden und 3 095 in verkaeufer, gefunden {zahl(len(kunden))} und '
          f'{zahl(len(verkaeufer))}')
elif (kunden['customer_zip_code_prefix'].astype(str).str.len() < 5).any():
    kurz = (kunden['customer_zip_code_prefix'].astype(str).str.len() < 5).sum()
    print(f'✗ prüfen: {zahl(kurz)} Postleitzahlen der Kunden haben weniger als 5 Stellen – die Null vorn ist weg. dtype beim Lesen')
elif (verkaeufer['seller_zip_code_prefix'].astype(str).str.len() < 5).any():
    print('✗ prüfen: bei den Verkäufern fehlen vorn Nullen – dort heißt die Spalte seller_zip_code_prefix')
else:
    print('✓ richtig · 99 441 Zeilen in kunden (eine je Bestellung), 3 095 Verkäufer, jede Postleitzahl mit 5 Stellen '
          '(z. B. 09790). Als Zahl gelesen hätten 23 995 Kundenzeilen und 1 027 Verkäufer ihre Null verloren.')

<details><summary><b>💡 Tipp</b> · erst selbst versuchen, dann aufklappen</summary>
<p>Komma als Trennzeichen und keine Dezimalzahlen: Dafür passt die Voreinstellung. Für die Postleitzahl: <code>dtype={'spalte': str}</code> liest eine Spalte als Text.</p>
</details>

## Aufgabe 3 · Web über eine API: der Euro-Kurs jedes Tages

**📖 Worum es geht.** Den Kurs jedes Tages gibt die brasilianische Zentralbank (Banco Central do Brasil) über eine API
heraus – frei, ohne API-Schlüssel. Die Kursreihe 21619 sagt: So viele Real kostet ein Euro. Wir brauchen September 2016
bis Oktober 2018; so weit reichen die Bestellungen. Die URL besteht wie in der Vorlesung aus **Endpoint** und
**Parametern**:

| Teil | Wert |
|---|---|
| Endpoint | `https://api.bcb.gov.br/dados/serie/bcdata.sgs.21619/dados` |
| Parameter | `formato=json` · `dataInicial=01/09/2016` · `dataFinal=31/10/2018` (Tag/Monat/Jahr) |

Dann wie in der Vorlesung: Statuscode prüfen, das JSON ansehen – wo stehen die Daten? Bei Open-Meteo lagen sie unter
`hourly`. Dann die Tabelle bauen. *Antwortet die API nicht:* Der Tipp nennt eine Sicherungskopie.

In [ ]:
# ===== Jetzt Sie (3) =====
url = None         # TODO: Endpoint + ? + die drei Parameter, mit & verbunden
antwort = None     # TODO: Anfrage schicken, dann den Statuscode prüfen
daten = None       # TODO: das JSON als Python – zuerst ansehen, z. B. daten[:2]
kurse_roh = None   # TODO: daraus eine Tabelle

# --- Check ---
if kurse_roh is None:
    print('… noch offen')
elif list(kurse_roh.columns) != ['data', 'valor']:
    print(f'✗ prüfen: erwartet die Spalten data und valor, gefunden {list(kurse_roh.columns)}')
elif len(kurse_roh) != 542:
    print(f'✗ prüfen: erwartet 542 Tage, gefunden {zahl(len(kurse_roh))} – stimmt der Zeitraum in der URL?')
else:
    print(f"✓ richtig · 542 Tage mit Kurs, der erste: '{kurse_roh.iloc[0, 0]}' mit '{kurse_roh.iloc[0, 1]}' R$ – "
          'beides noch Text.')

<details><summary><b>💡 Tipp</b> · erst selbst versuchen, dann aufklappen</summary>
<p><code>requests.get(url, timeout=60)</code>, dann <code>antwort.raise_for_status()</code> und <code>antwort.json()</code> – wie beim Wetter. Die Antwort ist schon eine Liste von Zeilen: <code>pd.DataFrame(daten)</code>. Sicherungskopie, falls die Zentralbank nicht antwortet: <code>daten = requests.get(REPO_URL + 'bcb/euro_real_2016_2018.json').json()</code></p>
</details>

---
# Block T · Transformieren

Aus den Rohdaten wird jetzt die Tabelle, die die Frage beantwortet: **eine Zeile je gelieferter Bestellung** – mit Staat,
„zu spät“, Umsatz in Real, Kurs am Kauftag und Umsatz in Euro. Dafür kommt alles aus der Vorlesung vor: bereinigen,
vereinheitlichen, zusammenführen (untereinander und nebeneinander), berechnen, aggregieren – und nach jedem Schritt
prüfen.

## Aufgabe 4 · Bereinigen und berechnen: Wann ist eine Bestellung zu spät?

**📖 Worum es geht.** Drei Schritte mit Werkzeugen aus Woche 5 – jetzt als Teil von T:

1. **Typen:** Die drei Zeitspalten sind Text. Erst als Datum lässt sich der Tag von der Uhrzeit trennen.
2. **Lücken:** Manche gelieferte Bestellung hat kein Lieferdatum. Ob sie zu spät kam, lässt sich nicht sagen – sie fällt
   weg. (Und man fragt bei Olist nach, warum.)
3. **Neue Spalten:** `spaet` ist 1, wenn die Bestellung **nach dem geplanten Tag** ankam, sonst 0. `kaufdatum` ist der
   Kauftag ohne Uhrzeit – über ihn kommt in Aufgabe 8 der Kurs.

**Vorsicht:** Geliefert ist ein Zeitpunkt (Tag und Uhrzeit), geplant nur ein Tag. Vergleichen Sie Tag mit Tag.

In [ ]:
# ===== Jetzt Sie (4) =====
# TODO 1: die drei Zeitspalten in bestellungen von Text in Datum umwandeln

# TODO 2: Bestellungen ohne Lieferdatum entfernen

# TODO 3: neue Spalten 'spaet' (1 = nach dem geplanten Tag geliefert, sonst 0) und 'kaufdatum' (Tag ohne Uhrzeit)

# --- Check ---
if bestellungen is None or 'spaet' not in bestellungen.columns or 'kaufdatum' not in bestellungen.columns:
    print('… noch offen')
elif not pd.api.types.is_datetime64_any_dtype(bestellungen['order_delivered_customer_date']):
    print('✗ prüfen: die Zeitspalten sind noch Text – pd.to_datetime')
elif bestellungen['order_delivered_customer_date'].isna().any():
    print(f"✗ prüfen: noch {bestellungen['order_delivered_customer_date'].isna().sum()} Bestellungen ohne Lieferdatum – dropna(subset=[…])")
elif len(bestellungen) != 96470:
    print(f'✗ prüfen: erwartet 96 470 Bestellungen, gefunden {zahl(len(bestellungen))}')
elif round(bestellungen['spaet'].mean(), 4) == 0.0811:
    print('✗ prüfen: 8,1 % zu spät – Sie vergleichen einen Zeitpunkt mit einem Tag. Wer am geplanten Tag beliefert wurde, '
          'zählt so als zu spät. Vergleichen Sie Tag mit Tag: .dt.date')
elif bestellungen['spaet'].sum() != 6534:
    print(f"✗ prüfen: erwartet 6 534 verspätete Bestellungen, gefunden {zahl(bestellungen['spaet'].sum())}")
elif pd.Series(pd.to_datetime(bestellungen['kaufdatum'])).dt.hour.any():
    print('✗ prüfen: kaufdatum soll der Tag ohne Uhrzeit sein – .dt.date')
else:
    print('✓ richtig · 8 Bestellungen ohne Lieferdatum entfernt, 96 470 bleiben. 6 534 kamen zu spät: 6,8 %.')

<details><summary><b>💡 Tipp</b> · erst selbst versuchen, dann aufklappen</summary>
<p><code>pd.to_datetime(spalte)</code> macht ein Datum, <code>.dt.date</code> lässt die Uhrzeit weg, <code>dropna(subset=['spalte'])</code> entfernt Zeilen mit Lücke in dieser Spalte. Ein Vergleich gibt True/False, <code>.astype(int)</code> macht 1/0 daraus.</p>
</details>

## Aufgabe 5 · Untereinander: Wo sitzen Kunden und Verkäufer?

**📖 Worum es geht.** Eine Vermutung zur Frage: Pakete kommen dort zu spät, wo kaum Verkäufer sitzen. Prüfen lässt sich
das mit **einer** Tabelle aller Orte: Kunden und Verkäufer untereinander, mit einer Kennspalte `rolle` – wie das Wetter
von Bottrop und Mülheim in der Vorlesung. Zwei Unterschiede:

- In `kunden` steht eine Zeile je **Bestellung**; ein Mensch kommt mehrfach vor. Der Mensch ist `customer_unique_id` –
  wie in Woche 5, Aufgabe 4.
- Die Spalten heißen verschieden (`customer_city`, `seller_city`). `concat` stellt nur Spalten mit **gleichem Namen**
  untereinander.

Ziel: `orte` mit genau den Spalten `plz`, `stadt`, `staat`, `rolle` (`'Kunde'` oder `'Verkäufer'`).

In [ ]:
# ===== Jetzt Sie (5) =====
orte = None    # TODO: Kunden (ein Mensch = eine Zeile) und Verkäufer untereinander – Spalten plz, stadt, staat, rolle

# --- Check ---
if orte is None:
    print('… noch offen')
elif set(orte.columns) != {'plz', 'stadt', 'staat', 'rolle'}:
    print(f'✗ prüfen: erwartet genau plz, stadt, staat, rolle – gefunden {list(orte.columns)}. Verschiedene Namen landen '
          'nebeneinander, mit Lücken: erst umbenennen, dann concat')
elif set(orte['rolle']) != {'Kunde', 'Verkäufer'}:
    print(f"✗ prüfen: rolle soll 'Kunde' oder 'Verkäufer' sein – gefunden {sorted(set(orte['rolle']))}")
elif len(orte) == 99441 + 3095:
    print('✗ prüfen: 102 536 Zeilen – die Kunden stehen noch mit einer Zeile je Bestellung darin. Ein Mensch = eine '
          'customer_unique_id')
elif len(orte) != 99191:
    print(f'✗ prüfen: erwartet 96 096 Kunden + 3 095 Verkäufer = 99 191 Zeilen, gefunden {zahl(len(orte))}')
else:
    print('✓ richtig · 99 191 Zeilen. São Paulo (SP): 42 % der Kunden, 60 % der Verkäufer · Rio de Janeiro (RJ): 13 % der '
          'Kunden, 5,5 % der Verkäufer · Alagoas (AL): kein einziger Verkäufer.')

<details><summary><b>💡 Tipp</b> · erst selbst versuchen, dann aufklappen</summary>
<p><code>drop_duplicates('customer_unique_id')</code>, dann bei beiden Tabellen <code>rename(columns={…})</code> auf dieselben Namen und die Spalte <code>rolle</code> anlegen, dann <code>pd.concat([…, …])</code> mit genau den vier Spalten.</p>
</details>

## Aufgabe 6 · Nebeneinander: Staat und Umsatz an jede Bestellung

**📖 Worum es geht.** Jetzt kommen Spalten dazu, über einen Schlüssel: `merge`. Die Praxisregel aus der Vorlesung:
**Wird rechts nachgeschlagen, steht dort jeder Schlüssel nur einmal – sonst vervielfacht merge die Zeilen, und Summen
zählen mehrfach.** Ist die rechte Tabelle feiner, mit mehreren Zeilen je Schlüssel: erst aggregieren, nicht löschen. Am Ende muss es eine Zeile je Bestellung bleiben, also so viele
Zeilen wie in `bestellungen`. Prüfen Sie das selbst mit `assert`.

- **6.1 Staat:** `customer_state` steht in `kunden`, Schlüssel `customer_id`. Welche Beziehung ist das, welches
  `validate` passt?
- **6.2 Umsatz:** Preis und Fracht stehen in `positionen` – eine Zeile je **Artikel**, und eine Bestellung kann mehrere
  Artikel haben. Ziel: je Bestellung `umsatz_brl` = Summe aus Preis und Fracht, in Real.

In [ ]:
# ===== Jetzt Sie (6.1) =====
mit_staat = None   # TODO: bestellungen + customer_state aus kunden, mit validate

# --- Check ---
if mit_staat is None:
    print('… noch offen')
elif 'customer_state' not in mit_staat.columns:
    print('✗ prüfen: customer_state fehlt – merge mit kunden über customer_id')
elif len(mit_staat) != 96470:
    print(f'✗ prüfen: erwartet 96 470 Zeilen wie in bestellungen, gefunden {zahl(len(mit_staat))}')
else:
    print("✓ richtig · 96 470 Zeilen, jede Bestellung hat ihren Staat. Jede customer_id gehört zu genau einer "
          "Bestellung und steht in kunden genau einmal: validate='one_to_one'.")

<details><summary><b>💡 Tipp</b> · erst selbst versuchen, dann aufklappen</summary>
<p><code>bestellungen.merge(kunden[[…]], on='customer_id', validate=…)</code>. Danach <code>assert len(mit_staat) == len(bestellungen)</code>.</p>
</details>

In [ ]:
# ===== Jetzt Sie (6.2) =====
je_bestellung = None    # TODO: aus positionen je order_id eine Zeile mit umsatz_brl (Preis + Fracht)
mit_positionen = None   # TODO: mit_staat + je_bestellung; danach die Zeilenzahl mit assert prüfen

# --- Check ---
if je_bestellung is None or mit_positionen is None:
    print('… noch offen')
elif len(mit_positionen) == 110189:
    print('✗ prüfen: 110 189 Zeilen statt 96 470 – in positionen stehen mehrere Zeilen je Bestellung, eine je Artikel. Erst je '
          'order_id aggregieren, dann verknüpfen.')
elif len(mit_positionen) != 96470:
    print(f'✗ prüfen: erwartet 96 470 Zeilen, gefunden {zahl(len(mit_positionen))}')
elif 'umsatz_brl' not in mit_positionen.columns or abs(mit_positionen['umsatz_brl'].sum() - 15418394.83) > 1:
    print('✗ prüfen: umsatz_brl = Preis + Fracht, je Bestellung summiert')
else:
    print('✓ richtig · 96 470 Zeilen, eine je Bestellung. 9 635 Bestellungen haben mehr als einen Artikel – direkt '
          'verknüpft wären daraus 110 189 Zeilen geworden.')

<details><summary><b>💡 Tipp</b> · erst selbst versuchen, dann aufklappen</summary>
<p>Erst eine Spalte Preis + Fracht in <code>positionen</code>, dann <code>groupby('order_id')[…].sum().reset_index()</code> – wie der Gewinn je Tag in der Vorlesung. Dann <code>merge</code> mit <code>validate='one_to_one'</code>.</p>
</details>

## Aufgabe 7 · validate und how: die Bewertungen

**📖 Worum es geht.** Zu jeder Bestellung gehört eine Bewertung von 1 bis 5 Sternen – so die Annahme. Prüfen Sie sie:
Verknüpfen Sie zuerst mit `validate='one_to_one'` und lesen Sie, was `validate` meldet. Hat eine Bestellung mehrere
Bewertungen, zählt ihr Mittelwert. Und: Nicht jede Bestellung wurde bewertet. Beim Verknüpfen darf keine Bestellung
verloren gehen.

In [ ]:
# ===== Jetzt Sie (7) =====
# TODO: erst mit_positionen und bewertungen mit validate='one_to_one' verknüpfen, die Meldung lesen,
#       dann Ihre Zeile mit # auskommentieren
noten = None       # TODO: je Bestellung eine Zeile – der Mittelwert ihrer review_score
mit_noten = None   # TODO: mit_positionen + noten – keine Bestellung darf fehlen

# --- Check ---
if noten is None or mit_noten is None:
    print('… noch offen')
elif noten['order_id'].duplicated().any():
    print('✗ prüfen: in noten steht eine Bestellung noch mehrfach – erst je order_id aggregieren')
elif len(mit_noten) == 95824:
    print("✗ prüfen: 95 824 Zeilen – 646 Bestellungen ohne Bewertung sind verloren gegangen. how='left' behält alle "
          "Bestellungen der linken Tabelle.")
elif len(mit_noten) != 96470:
    print(f'✗ prüfen: erwartet 96 470 Zeilen, gefunden {zahl(len(mit_noten))}')
else:
    print('✓ richtig · 96 470 Zeilen. 525 Bestellungen hatten 2 bis 3 Bewertungen, bei 189 mit verschiedenen Sternen – '
          'jetzt der Mittelwert. 646 ohne Bewertung: Note leer. Zu spät im Schnitt 2,3 Sterne, pünktlich 4,3.')

<details><summary><b>💡 Tipp</b> · erst selbst versuchen, dann aufklappen</summary>
<p>In <code>bewertungen</code> stehen mehrere Zeilen je Bestellung: erst je <code>order_id</code> aggregieren, hier mit dem Mittelwert. Dann <code>merge(…, how='left', validate='one_to_one')</code>.</p>
</details>

## Aufgabe 8 · Der Kurs am Kauftag

**📖 Worum es geht.** Jetzt der Euro, in zwei Schritten.

**8.1 Schlüssel vereinheitlichen, verknüpfen, prüfen.** Der Schlüssel ist der Kauftag. In `kurse_roh` heißt er `data` und
ist Text (`01/09/2016`), der Kurs `valor` ebenfalls Text. In `mit_noten` heißt er `kaufdatum` und ist ein Tag aus Aufgabe
4. Name, Typ und Granularität müssen gleich sein – wie bei der Artikelnummer `00042` in der Vorlesung. Dann verknüpfen, alle
Bestellungen behalten und mit `indicator=True` zählen: Wie viele Bestellungen finden keinen Kurs – und an welchen Tagen?

**8.2 Lücken füllen und umrechnen.** Die Zentralbank veröffentlicht nur an Bankarbeitstagen einen Kurs. Unsere Regel –
eine Regel der T-Phase, wie die Höchsttemperatur je Tag in der Vorlesung: Fehlt der Tageskurs, gilt der **Durchschnitt
des Monats**.
Das ist das Füllen je Gruppe aus Woche 5; verknüpft wird über **zwei** Schlüsselspalten, `jahr` und `monat`, wie
Filiale und Datum in der Vorlesung. Dann `umsatz_eur = umsatz_brl / kurs`.

In [ ]:
# ===== Jetzt Sie (8.1) =====
kurse = None      # TODO: aus kurse_roh die Spalten kaufdatum (Tag) und kurs (Zahl), dazu jahr und monat
mit_kurs = None   # TODO: mit_noten + kurse[['kaufdatum', 'kurs']] – alle Bestellungen behalten, indicator=True

# --- Check ---
if kurse is None or mit_kurs is None:
    print('… noch offen')
elif '_merge' not in mit_kurs.columns:
    print('✗ prüfen: die Spalte _merge fehlt – indicator=True')
elif len(mit_kurs) != 96470:
    print(f"✗ prüfen: erwartet 96 470 Zeilen, gefunden {zahl(len(mit_kurs))} – alle Bestellungen behalten: how='left'")
elif (mit_kurs['_merge'] == 'both').sum() == 0:
    print('✗ prüfen: keine Bestellung hat einen Kurs gefunden – kaufdatum muss in beiden Tabellen gleich sein: '
          'Datum statt Text, Tag ohne Uhrzeit (.dt.date)')
elif (mit_kurs['_merge'] == 'left_only').sum() != 24270:
    print(f"✗ prüfen: erwartet 24 270 Bestellungen ohne Kurs, gefunden {zahl((mit_kurs['_merge'] == 'left_only').sum())}")
elif not {'jahr', 'monat'}.issubset(kurse.columns) or not pd.api.types.is_numeric_dtype(kurse['kurs']):
    print('✗ prüfen: in kurse fehlen jahr und monat, oder kurs ist noch Text – astype(float)')
else:
    print('✓ richtig · 72 200 Bestellungen mit Tageskurs, 24 270 ohne: 22 187 am Wochenende, 2 083 an Feiertagen, '
          'z. B. Karneval am 12. und 13.02.2018 mit 431 Bestellungen.')

<details><summary><b>💡 Tipp</b> · erst selbst versuchen, dann aufklappen</summary>
<p><code>rename</code> für den Namen, <code>astype(float)</code> und <code>pd.to_datetime(…, format='%d/%m/%Y')</code> für den Typ. Vom Datum zuerst <code>.dt.year</code> und <code>.dt.month</code>, zuletzt <code>.dt.date</code> für den Tag – nach <code>.dt.date</code> gibt es kein <code>.dt</code> mehr. Dann nur <code>kurse[['kaufdatum', 'kurs']]</code> mit <code>merge(…, how='left', validate='many_to_one', indicator=True)</code>.</p>
</details>

In [ ]:
# ===== Jetzt Sie (8.2) =====
je_monat = None  # TODO: Durchschnittskurs je jahr und monat, aus kurse – Spalte kurs_monat
in_euro = None   # TODO: in mit_kurs jahr und monat aus order_purchase_timestamp anlegen, mit je_monat verknüpfen,
                 #       Lücken in kurs mit kurs_monat füllen, neue Spalte umsatz_eur; zum Schluss mit assert prüfen

# --- Check ---
if je_monat is None or in_euro is None:
    print('… noch offen')
elif len(je_monat) != 26:
    print(f"✗ prüfen: erwartet 26 Monate (September 2016 bis Oktober 2018), gefunden {zahl(len(je_monat))} – "
          "groupby(['jahr', 'monat'])")
elif len(in_euro) != 96470:
    print(f'✗ prüfen: erwartet 96 470 Zeilen, gefunden {zahl(len(in_euro))}')
elif in_euro['kurs'].isna().any():
    print(f"✗ prüfen: noch {zahl(in_euro['kurs'].isna().sum())} Bestellungen ohne Kurs – fillna mit kurs_monat")
elif 'umsatz_eur' not in in_euro.columns or abs(in_euro['umsatz_eur'].sum() - 3906441.79) > 50:
    print('✗ prüfen: umsatz_eur = umsatz_brl / kurs')
else:
    print('✓ richtig · Alle 96 470 Bestellungen haben einen Kurs; zusammen 3,91 Mio. € Umsatz. Der Tageskurs '
          'zählt, wo es ihn gibt: Innerhalb eines Monats schwankte der Kurs um bis zu 13 % (August 2018).')

<details><summary><b>💡 Tipp</b> · erst selbst versuchen, dann aufklappen</summary>
<p>Wie der Gewinn je Tag: <code>groupby(['jahr', 'monat'])['kurs'].mean().reset_index(name='kurs_monat')</code>. Dann <code>merge(…, on=['jahr', 'monat'])</code> und <code>spalte.fillna(andere_spalte)</code> wie in Woche 5.</p>
</details>

---
# Block L · Laden

## Aufgabe 9 · In die Projekt-Datenbank

**📖 Worum es geht.** Die Tabelle steht jetzt nur im Arbeitsspeicher – Notebook zu, Tabelle weg. Also kommt sie in
eine Datenbank: `app_daten/projekt.db`, wie im Projekt. (In Colab verschwindet auch diese Datei, wenn die Sitzung endet:
links über das Ordner-Symbol herunterladen.) Hinein kommt, was Frage und Webseite brauchen, mit kurzen Namen:

| aus `in_euro` | `order_id` | `kaufdatum` | `customer_state` | `spaet` | `umsatz_brl` | `kurs` | `umsatz_eur` | `review_score` |
|---|---|---|---|---|---|---|---|---|
| **in der Datenbank** | `order_id` | `kaufdatum` | `staat` | `spaet` | `umsatz_brl` | `kurs` | `umsatz_eur` | `note` |

Dazu die Tabelle `orte` aus Aufgabe 5. Und `if_exists`: Im Projekt läuft der Code immer wieder. Mit denselben Daten
soll jeder Lauf dasselbe Ergebnis liefern. Welcher Wert sorgt dafür, dass nach dem zweiten Lauf dasselbe in der
Datenbank steht wie nach dem ersten? Tragen Sie ihn bei `modus` ein; der Check lädt ein zweites Mal und zählt.

In [ ]:
# ===== Jetzt Sie (9) =====
os.makedirs('app_daten', exist_ok=True)       # der Ordner wie in Ihrem Projekt
projekt = None    # TODO: Verbindung zu app_daten/projekt.db
ergebnis = None   # TODO: aus in_euro die acht Spalten aus der Tabelle oben, mit den neuen Namen
modus = None      # TODO: 'fail', 'replace' oder 'append'?
# TODO: ergebnis als Tabelle 'bestellungen' und orte als Tabelle 'orte' laden – if_exists=modus, index=False

# --- Check: lädt ein zweites Mal, wie beim nächsten Lauf Ihres Codes ---
if projekt is None or ergebnis is None or modus is None:
    print('… noch offen')
elif list(ergebnis.columns) != ['order_id', 'kaufdatum', 'staat', 'spaet', 'umsatz_brl', 'kurs', 'umsatz_eur', 'note']:
    print(f'✗ prüfen: erwartet die acht Spalten aus der Tabelle oben, in dieser Reihenfolge – gefunden {list(ergebnis.columns)}')
elif not {'bestellungen', 'orte'} <= set(pd.read_sql("SELECT name FROM sqlite_master WHERE type = 'table'", projekt)['name']):
    print("✗ prüfen: in projekt.db fehlt noch eine Tabelle – erst selbst laden: to_sql für 'bestellungen' und 'orte'")
else:
    try:
        ergebnis.to_sql('bestellungen', projekt, if_exists=modus, index=False)
        zeilen = pd.read_sql('SELECT COUNT(*) AS n FROM bestellungen', projekt)['n'].iloc[0]
    except ValueError:
        zeilen = None
    if zeilen is None:
        print("✗ prüfen: Der zweite Lauf bricht ab, weil es die Tabelle schon gibt – 'fail' taugt nur für den ersten Lauf.")
    elif zeilen != len(ergebnis):
        ergebnis.to_sql('bestellungen', projekt, if_exists='replace', index=False)     # aufräumen
        print(f'✗ prüfen: Nach dem zweiten Lauf standen {zahl(zeilen)} Zeilen in der Datenbank – alles doppelt.')
    else:
        print("✓ richtig · Zweimal geladen, 96 470 Zeilen – wie nach dem ersten Lauf. 'replace' ersetzt die alte Tabelle: "
              'Der Code darf beliebig oft laufen – idempotent, wie in der Vorlesung.')

<details><summary><b>💡 Tipp</b> · erst selbst versuchen, dann aufklappen</summary>
<p><code>sqlite3.connect('app_daten/projekt.db')</code> legt die Datei an. Spalten auswählen mit <code>tabelle[[…]]</code>, umbenennen mit <code>rename(columns={…})</code>. Dann <code>to_sql</code> wie in der Vorlesung.</p>
</details>

## Aufgabe 10 · Die Frage beantworten – mit SQL

**📖 Worum es geht.** Die Daten liegen jetzt in einer Datenbank; die Antwort holen Sie mit SQL – `GROUP BY` kennen
Sie aus der Datenbank-Vorlesung. Schreiben Sie **eine** Abfrage auf `bestellungen`: je `staat` die Zahl der Bestellungen, den Anteil
verspäteter Bestellungen in Prozent (eine Nachkommastelle) und den verspäteten Umsatz in Euro (gerundet) – sortiert nach
dem verspäteten Umsatz, der größte zuerst.

Zwei Hilfen: `spaet` ist 1 oder 0, also ist der Durchschnitt von `spaet` der Anteil, mal 100 in Prozent. Und
`spaet * umsatz_eur` ist der
Umsatz einer verspäteten Bestellung, bei einer pünktlichen 0.

In [ ]:
# ===== Jetzt Sie (10) =====
sql = None        # TODO: die Abfrage als Text
je_staat = None   # TODO: pd.read_sql(…) auf projekt

# --- Check ---
if je_staat is None:
    print('… noch offen')
else:
    text_spalten = [s for s in je_staat.columns if je_staat[s].astype(str).isin(['SP']).any()]
    zahlen = je_staat.select_dtypes('number')
    if not text_spalten or len(je_staat) != 27:
        print(f'✗ prüfen: erwartet eine Zeile je Staat, also 27 Zeilen – gefunden {zahl(len(je_staat))}. GROUP BY staat')
    elif list(je_staat[text_spalten[0]].iloc[:2]) != ['SP', 'RJ']:
        print('✗ prüfen: sortiert nach dem verspäteten Umsatz, der größte zuerst, stehen oben SP und RJ – ORDER BY … DESC')
    elif not any((zahlen[s].iloc[:2] - [76318, 64222]).abs().max() <= 2 for s in zahlen.columns):
        print('✗ prüfen: der verspätete Umsatz für SP und RJ stimmt nicht – erwartet rund 76 318 und 64 222 €. '
              'SUM(spaet * umsatz_eur)')
    elif not any(abs(zahlen[s].iloc[1] - 12.1) < 0.06 or abs(zahlen[s].iloc[1] - 0.121) < 0.0006 for s in zahlen.columns):
        print('✗ prüfen: der Anteil verspäteter Bestellungen in RJ stimmt nicht – erwartet 12,1 %. '
              'ROUND(100.0 * AVG(spaet), 1)')
    else:
        print('✓ richtig · Der meiste verspätete Umsatz: SP 76 318 €, RJ 64 222 € – RJ mit nicht einmal einem Drittel der '
              'Bestellungen (12 350 gegen 40 494). Am häufigsten zu spät: AL 21,4 %, MA 17,4 %.')

<details><summary><b>💡 Tipp</b> · erst selbst versuchen, dann aufklappen</summary>
<p><code>COUNT(*)</code>, <code>ROUND(100.0 * AVG(spaet), 1)</code>, <code>ROUND(SUM(spaet * umsatz_eur))</code>; dann <code>GROUP BY staat</code> und <code>ORDER BY … DESC</code>.</p>
</details>

**Ihre Antwort auf die Frage (zwei bis drei Sätze):** Wo kommen Bestellungen am häufigsten zu spät, wo steckt der meiste verspätete Umsatz in Euro – und was sagt die Tabelle `orte` aus Aufgabe 5 dazu?

*Ihre Antwort:*

---
# ⭐ Aufgaben für einzelne Projekt-Karten

## ⭐ Aufgabe 11 · Dieselben Kurse als CSV (Karte 2)

**📖 Worum es geht.** Quelle und Format sind zwei Fragen. Die Quelle bleibt die API der Zentralbank; mit `formato=csv`
liefert sie dasselbe als CSV. Die Zelle unten zeigt den Anfang. Erst ansehen: Welches Trennzeichen, welches
Dezimalzeichen? Karte 2 holt den Euribor ebenfalls als CSV über eine API (EZB, `format=csvdata`) – auch dort erst
ansehen, welche Zeichen die Datei benutzt.

In [ ]:
# ===== Erst ansehen: der Anfang der CSV-Antwort =====
url_csv = ('https://api.bcb.gov.br/dados/serie/bcdata.sgs.21619/dados'
           '?formato=csv&dataInicial=01/09/2016&dataFinal=31/10/2018')
print(requests.get(url_csv, timeout=60).text[:80])

In [ ]:
# ===== Jetzt Sie (⭐ 11) =====
kurse_csv = None   # TODO: url_csv mit pd.read_csv lesen – so, dass valor gleich eine Zahl wird

# --- Check ---
if kurse_csv is None:
    print('… noch offen')
elif kurse_csv.shape[1] != 2:
    print(f'✗ prüfen: erwartet zwei Spalten, gefunden {kurse_csv.shape[1]} – welches Trennzeichen steht in der Datei?')
elif not pd.api.types.is_float_dtype(kurse_csv['valor']):
    print('✗ prüfen: valor ist noch Text – welches Dezimalzeichen steht in der Datei?')
else:
    print(f"✓ richtig · {zahl(len(kurse_csv))} Kurse, valor ist gleich eine Zahl ({kurse_csv['valor'].iloc[0]}). Gleiche "
          'Quelle, anderes Format, anderes Lesen.')

<details><summary><b>💡 Tipp</b> · erst selbst versuchen, dann aufklappen</summary>
<p>Wie <code>artikel.csv</code> in der Vorlesung: <code>sep=';'</code> und <code>decimal=','</code>.</p>
</details>

## ⭐ Aufgabe 12 · Dieselbe Nachschlagetabelle zweimal (Karte 6, Karte 3)

**📖 Worum es geht.** Karte 6 verknüpft die Zonentabelle zweimal: einmal für den Abholort, einmal für das Ziel. Hier
dasselbe Muster mit einer Tabelle der 27 Bundesstaaten und ihrer Großregion (Norte, Nordeste, Centro-Oeste, Sudeste,
Sul), frei auf GitHub – Karte 3 liest ihre Ländertabelle genauso direkt von dort. Je **Position** brauchen wir die Region
des Kunden und die Region des Verkäufers. Damit sich die zwei Region-Spalten nicht in die Quere kommen, benennen Sie die
Nachschlagetabelle vor jedem `merge` passend um. Frage: Bei wie vielen Positionen verlässt das Paket seine Region – und
kommen diese öfter zu spät?

Der Weg: `order_items` (`order_id`, `seller_id`) → `verkaeufer` (`seller_state`) → `mit_staat` (`customer_state`,
`spaet`) → zweimal `regionen`.

In [ ]:
# ===== Jetzt Sie (⭐ 12) =====
regionen = pd.read_csv('https://raw.githubusercontent.com/kelvins/municipios-brasileiros/main/csv/estados.csv')[['uf', 'regiao']]   # fertig: Staat → Region
paket = None   # TODO: order_items (order_id, seller_id) neu aus olist.db lesen, Staat des Verkäufers und über mit_staat
               #       Staat des Kunden und spaet anfügen (nur gelieferte Bestellungen bleiben); dann zweimal regionen –
               #       dabei darf keine Position verloren gehen (assert)

# --- Check ---
if paket is None:
    print('… noch offen')
elif not {'region_kunde', 'region_verkaeufer', 'spaet'}.issubset(paket.columns):
    print('✗ prüfen: es fehlen region_kunde, region_verkaeufer oder spaet – zweimal mit regionen verknüpfen, vorher umbenennen')
elif len(paket) != 110189:
    print(f'✗ prüfen: erwartet 110 189 Positionen gelieferter Bestellungen, gefunden {zahl(len(paket))}')
else:
    print('✓ richtig · 110 189 Positionen, 38 % verlassen ihre Region. Von ihnen kommen 7,5 % zu spät, '
          'innerhalb der Region 6,0 %.')

<details><summary><b>💡 Tipp</b> · erst selbst versuchen, dann aufklappen</summary>
<p><code>positionen</code> hat keine <code>seller_id</code> – also <code>pd.read_sql</code> auf <code>order_items</code>. Erst die zwei Staaten an jede Position; der merge mit <code>mit_staat</code> behält nur gelieferte Bestellungen. Dann <code>n = len(paket)</code>, zweimal <code>regionen.rename(columns={'uf': …, 'regiao': …})</code> mit <code>merge</code> und zum Schluss <code>assert len(paket) == n</code>.</p>
</details>

---
# Und jetzt Ihr Projekt

Schritt 4 und 5 Ihres Projekts sind genau dieser Weg – mit Ihrer Karte.

1. **Erst planen, dann programmieren.** Schreiben Sie für Ihre Karte fünf Zeilen auf:
   - **E:** Woher kommen die Rohdaten, woher die zweite Quelle – Datei, API oder Datenbank? Was holen Sie davon?
   - **T:** Was ist der Schlüssel? Ist er in beiden Tabellen gleich – Name, Typ, Granularität?
   - **T:** Hat die zweite Quelle eine Zeile je Schlüssel, oder müssen Sie erst aggregieren?
   - **T:** Welche Zeilen müssen bleiben (`how`)? Wie prüfen Sie danach (`assert`, `indicator=True`)?
   - **L:** Welche Tabelle mit welchen Spalten kommt in `app_daten/projekt.db`?
2. **Das Ablaufbild (Schritt 5)** zeichnen Sie aus diesen Zeilen: je Station ein Kästchen, dazwischen Pfeile. Für Olist:

```
E   olist.db  ·  Kunden- und Verkäufer-CSV  ·  Zentralbank-API
     │
     ▼
T   bereinigen · vereinheitlichen · zusammenführen · berechnen · aggregieren
     │
     ▼
L   app_daten/projekt.db
     │
     ▼
    Webseite
```

3. **Hinweise zu einzelnen Karten:** Karte 1 – stündliche Werte: Datum und Stunde als zwei Schlüsselspalten, wie
   `jahr` und `monat` in Aufgabe 8. Karte 4 – `pd.read_csv` liest `calendar.csv.gz` direkt, ohne Entpacken. Karte 5 – in
   `IDS_mapping.csv` stehen drei Tabellen untereinander: Lesen Sie jede einzeln, mit `skiprows` (Zeilen vorn überspringen)
   und `nrows` (nur so viele Zeilen lesen).